# 26 — Human-Centred AI and Trust Calibration

    ## Scenario and success criteria

    An industrial-support answer with high claimed confidence still requires human review when it proposes bypassing a safety control.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Calibrate confidence against outcomes.
- Tie escalation to impact and evidence.
- Expose user-facing uncertainty without hidden reasoning.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 26 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Self-reported confidence is not permission and can be badly calibrated.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab26 import ProposedAnswer, calibration_error, decide_delivery

known = {"manual-X900-v4"}
safe = ProposedAnswer("Use the documented reset procedure.", 0.9, ("manual-X900-v4",), "low")
dangerous = ProposedAnswer("Bypass the regulator.", 0.99, ("manual-X900-v4",), "high")
unsupported = ProposedAnswer("Change internal wiring.", 0.95, ("invented",), "medium")

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
for answer in (safe, dangerous, unsupported):
    print(answer.response, "=>", decide_delivery(answer, known))
print("calibration error", calibration_error([(0.9, True), (0.8, False), (0.6, True)]))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert decide_delivery(safe, known).state == "answer"
assert decide_delivery(dangerous, known).state == "human_review"
assert decide_delivery(unsupported, known).state == "blocked"
assert 0 <= calibration_error([(0.9, True), (0.8, False)]) <= 1

## Production upgrade

Design review queues with ownership and service levels, show sources and limitations to users, measure override and harm outcomes, prevent automation bias, and never log private chain-of-thought as an explanation.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.